# Week 7: FT-Transformer (Multi-Output) — Cross-Disease Learning

**Goal:** Train a multi-output FT-Transformer on masked data to model interrelations between heart, diabetes, liver, and kidney risks.

**Config source:** `configs/week7_ft_transformer.yaml`

## 1) Setup & Reproducibility

In [ ]:
import os, json, random, time
from contextlib import nullcontext
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

import yaml

try:
    from IPython.display import display as ipy_display
except Exception:
    ipy_display = None


def show_df(df: pd.DataFrame, max_rows: int = 30):
    if ipy_display is not None:
        ipy_display(df)
    else:
        print(df.head(max_rows).to_string(index=False))


def autocast_context(enabled: bool):
    if torch.cuda.is_available():
        return torch.amp.autocast(device_type='cuda', enabled=enabled)
    return nullcontext()


def find_project_root(start: Path) -> Path:
    """Find repo root by locating configs/week7_ft_transformer.yaml in current or parent dirs."""
    probe = start.resolve()
    for candidate in [probe, *probe.parents]:
        cfg_path = candidate / 'configs' / 'week7_ft_transformer.yaml'
        if cfg_path.exists():
            return candidate
    raise FileNotFoundError('Could not locate configs/week7_ft_transformer.yaml from current working directory.')


PROJECT_ROOT = find_project_root(Path.cwd())
CONFIG_PATH = PROJECT_ROOT / 'configs' / 'week7_ft_transformer.yaml'

with open(CONFIG_PATH, 'r', encoding='utf-8') as f:
    CFG = yaml.safe_load(f)

SEED = int(CFG['experiment']['random_seed'])
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

if bool(CFG['experiment'].get('deterministic_mode', False)):
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print(f"Project root: {PROJECT_ROOT}")
print(f"Config loaded: {CONFIG_PATH}")
print(f"Torch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"Device: {DEVICE}")

ModuleNotFoundError: No module named 'numpy'

## 2) Load Data Splits

In [ ]:
train_path = PROJECT_ROOT / CFG['data']['train_file']
val_path = PROJECT_ROOT / CFG['data']['val_file']
test_path = PROJECT_ROOT / CFG['data']['test_file']

for p in [train_path, val_path, test_path]:
    if not p.exists():
        raise FileNotFoundError(f"Missing split file: {p}")

df_train = pd.read_csv(train_path)
df_val = pd.read_csv(val_path)
df_test = pd.read_csv(test_path)

if list(df_train.columns) != list(df_val.columns) or list(df_train.columns) != list(df_test.columns):
    raise ValueError('Train/Val/Test columns are not aligned. Please fix split schema consistency.')

print('Train:', df_train.shape)
print('Val  :', df_val.shape)
print('Test :', df_test.shape)
print('Columns:', len(df_train.columns))

## 3) Build Feature & Target Columns

In [ ]:
target_map = CFG['data']['target_columns']
target_cols = list(target_map.values())
source_col = CFG['data']['source_column']
exclude_cols = set(CFG['data']['feature_policy']['exclude_columns'])

required_cols = set(target_cols + [source_col])
missing_required = required_cols - set(df_train.columns)
if missing_required:
    raise ValueError(f"Required columns missing in split files: {sorted(missing_required)}")

# Target sanitization: ensure dense numeric 0/1 matrix for multi-output training.
for frame_name, frame in [('train', df_train), ('val', df_val), ('test', df_test)]:
    frame[target_cols] = frame[target_cols].apply(pd.to_numeric, errors='coerce').fillna(0.0)
    frame[target_cols] = frame[target_cols].clip(lower=0.0, upper=1.0)

feature_cols = [c for c in df_train.columns if c not in exclude_cols]

# Coerce features to numeric and fill residual NaNs (masked pipeline should already be numeric).
for frame in [df_train, df_val, df_test]:
    frame[feature_cols] = frame[feature_cols].apply(pd.to_numeric, errors='coerce')

nan_train = int(df_train[feature_cols].isna().sum().sum())
nan_val = int(df_val[feature_cols].isna().sum().sum())
nan_test = int(df_test[feature_cols].isna().sum().sum())
if (nan_train + nan_val + nan_test) > 0:
    print(f"Warning: feature NaNs detected (train={nan_train}, val={nan_val}, test={nan_test}). Filling with 0.0")
    for frame in [df_train, df_val, df_test]:
        frame[feature_cols] = frame[feature_cols].fillna(0.0)

mask_cols = [c for c in feature_cols if c.endswith('_mask')]

print('No. of features:', len(feature_cols))
print('No. of targets :', len(target_cols))
print('No. of mask cols:', len(mask_cols))
print('Targets        :', target_cols)
print('Source distribution (train):', df_train[source_col].value_counts().to_dict())
print('Target prevalence (train):', df_train[target_cols].mean().round(4).to_dict())

## 4) PyTorch Dataset & DataLoader

In [ ]:
class MultiDiseaseDataset(Dataset):
    def __init__(self, df, feature_cols, target_cols):
        X = df[feature_cols].astype(np.float32).to_numpy(copy=True)
        y = df[target_cols].astype(np.float32).to_numpy(copy=True)
        self.X = np.ascontiguousarray(X)
        self.y = np.ascontiguousarray(y)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return torch.from_numpy(self.X[idx]), torch.from_numpy(self.y[idx])


profile_name = 'colab_gpu' if torch.cuda.is_available() else 'local_cpu'
profile = CFG['runtime_profiles'][profile_name]

batch_size = int(profile['batch_size'])
num_workers = int(profile['num_workers'])
pin_memory = bool(torch.cuda.is_available())

g = torch.Generator()
g.manual_seed(SEED)

train_ds = MultiDiseaseDataset(df_train, feature_cols, target_cols)
val_ds = MultiDiseaseDataset(df_val, feature_cols, target_cols)
test_ds = MultiDiseaseDataset(df_test, feature_cols, target_cols)

train_loader = DataLoader(
    train_ds,
    batch_size=batch_size,
    shuffle=True,
    num_workers=num_workers,
    pin_memory=pin_memory,
    drop_last=False,
    generator=g,
    persistent_workers=(num_workers > 0),
)
val_loader = DataLoader(
    val_ds,
    batch_size=batch_size,
    shuffle=False,
    num_workers=num_workers,
    pin_memory=pin_memory,
    drop_last=False,
    persistent_workers=(num_workers > 0),
)
test_loader = DataLoader(
    test_ds,
    batch_size=batch_size,
    shuffle=False,
    num_workers=num_workers,
    pin_memory=pin_memory,
    drop_last=False,
    persistent_workers=(num_workers > 0),
)

xb, yb = next(iter(train_loader))
print('Loaders ready.')
print('Runtime profile:', profile_name)
print('Batch size:', batch_size, '| Workers:', num_workers)
print('Batch X shape:', tuple(xb.shape), '| dtype:', xb.dtype)
print('Batch y shape:', tuple(yb.shape), '| dtype:', yb.dtype)
print('Device profile:', 'GPU-compatible' if pin_memory else 'CPU')

## 5) Model Skeleton (Multi-Output FT-Transformer)

In [ ]:
class MultiOutputFTTransformer(nn.Module):
    """
    FT-style tabular transformer with:
      - learned per-feature tokenization
      - learned disease query tokens (one per task)
      - shared transformer encoder
      - separate task heads
    """

    def __init__(
        self,
        n_features,
        n_outputs=4,
        d_token=64,
        n_heads=8,
        n_layers=3,
        ff_mult=4,
        attn_dropout=0.1,
        ff_dropout=0.1,
        out_dropout=0.1,
    ):
        super().__init__()

        if d_token % n_heads != 0:
            raise ValueError(f"d_token ({d_token}) must be divisible by n_heads ({n_heads})")

        self.n_features = n_features
        self.n_outputs = n_outputs
        self.d_token = d_token

        self.feature_weight = nn.Parameter(torch.empty(n_features, d_token))
        self.feature_bias = nn.Parameter(torch.empty(n_features, d_token))

        self.disease_tokens = nn.Parameter(torch.empty(1, n_outputs, d_token))

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_token,
            nhead=n_heads,
            dim_feedforward=d_token * ff_mult,
            dropout=ff_dropout,
            batch_first=True,
            activation='gelu',
            norm_first=True,
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=n_layers)

        self.token_dropout = nn.Dropout(attn_dropout)
        self.final_norm = nn.LayerNorm(d_token)

        self.heads = nn.ModuleList([
            nn.Sequential(
                nn.LayerNorm(d_token),
                nn.Dropout(out_dropout),
                nn.Linear(d_token, 1),
            )
            for _ in range(n_outputs)
        ])

        self.reset_parameters()

    def reset_parameters(self):
        nn.init.xavier_uniform_(self.feature_weight)
        nn.init.zeros_(self.feature_bias)
        nn.init.normal_(self.disease_tokens, mean=0.0, std=0.02)

    def feature_tokenize(self, x):
        # x: [B, F] -> tokens: [B, F, D]
        return x.unsqueeze(-1) * self.feature_weight.unsqueeze(0) + self.feature_bias.unsqueeze(0)

    def forward(self, x):
        # Feature tokens
        feat_tokens = self.feature_tokenize(x)

        # Disease query tokens (one token per target)
        B = x.shape[0]
        disease_tokens = self.disease_tokens.expand(B, -1, -1)

        # Concatenate [disease tokens | feature tokens]
        tokens = torch.cat([disease_tokens, feat_tokens], dim=1)
        tokens = self.token_dropout(tokens)

        encoded = self.encoder(tokens)
        encoded = self.final_norm(encoded)

        # First n_outputs tokens correspond to disease queries
        disease_repr = encoded[:, :self.n_outputs, :]  # [B, O, D]

        logits = []
        for i, head in enumerate(self.heads):
            logits.append(head(disease_repr[:, i, :]))
        return torch.cat(logits, dim=1)  # [B, O]


model_cfg = {
    'd_token': 64,
    'n_heads': 8,
    'n_layers': 3,
    'ff_mult': 4,
    'attn_dropout': 0.10,
    'ff_dropout': 0.10,
    'out_dropout': 0.10,
}

model = MultiOutputFTTransformer(
    n_features=len(feature_cols),
    n_outputs=len(target_cols),
    **model_cfg,
)

n_params = sum(p.numel() for p in model.parameters())
print(model.__class__.__name__, 'initialized.')
print('Model config:', model_cfg)
print('Trainable parameters:', f"{n_params:,}")
print('Outputs:', target_cols)

## 6) Training Loop Skeleton

In [ ]:
# Step 5: High-quality training loop for Colab GPU (also works on CPU)
from copy import deepcopy
from datetime import datetime, timezone

model = model.to(DEVICE)

# Task-wise class imbalance handling (pos_weight for BCEWithLogits)
pos_counts = df_train[target_cols].sum(axis=0).to_numpy(dtype=np.float32)
neg_counts = (len(df_train) - pos_counts).astype(np.float32)
pos_weight_np = np.clip(neg_counts / np.clip(pos_counts, 1.0, None), 1.0, 100.0)
pos_weight = torch.tensor(pos_weight_np, dtype=torch.float32, device=DEVICE)

criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=float(CFG['training']['learning_rate']),
    weight_decay=float(CFG['training']['weight_decay']),
)

max_epochs = int(CFG['training']['max_epochs'])
patience = int(CFG['training']['early_stopping_patience'])
monitor_metric = str(CFG['training']['monitor_metric'])
monitor_mode = str(CFG['training']['monitor_mode'])

# Cosine LR helps stabilize late-stage convergence
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=max_epochs)

# Mixed precision for GPU speed/throughput
use_amp = bool(torch.cuda.is_available()) and bool(CFG['runtime_profiles']['colab_gpu']['mixed_precision'])
scaler = torch.cuda.amp.GradScaler(enabled=use_amp)

# Artifact paths
model_dir = PROJECT_ROOT / CFG['artifacts']['model_dir']
metrics_dir = PROJECT_ROOT / CFG['artifacts']['metrics_dir']
model_dir.mkdir(parents=True, exist_ok=True)
metrics_dir.mkdir(parents=True, exist_ok=True)

best_ckpt_path = model_dir / 'best_model.pt'
run_meta_path = metrics_dir / 'run_metadata.json'
train_log_path = metrics_dir / 'train_history.csv'

def train_one_epoch(model, loader, optimizer, criterion, device, scaler, use_amp=True, grad_clip=1.0):
    model.train()
    running_loss = 0.0
    n_samples = 0

    for xb, yb in loader:
        xb = xb.to(device, non_blocking=True)
        yb = yb.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        with autocast_context(use_amp):
            logits = model(xb)
            loss = criterion(logits, yb)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=grad_clip)
        scaler.step(optimizer)
        scaler.update()

        bs = xb.size(0)
        running_loss += loss.detach().item() * bs
        n_samples += bs

    return running_loss / max(n_samples, 1)


@torch.no_grad()
def evaluate_loss(model, loader, criterion, device, use_amp=True):
    model.eval()
    running_loss = 0.0
    n_samples = 0

    for xb, yb in loader:
        xb = xb.to(device, non_blocking=True)
        yb = yb.to(device, non_blocking=True)

        with autocast_context(use_amp):
            logits = model(xb)
            loss = criterion(logits, yb)

        bs = xb.size(0)
        running_loss += loss.detach().item() * bs
        n_samples += bs

    return running_loss / max(n_samples, 1)


def is_better(curr, best, mode='min'):
    return curr < best if mode == 'min' else curr > best


history = []
best_state = None
best_val = float('inf') if monitor_mode == 'min' else -float('inf')
best_epoch = -1
wait = 0

print('Training config')
print('  Device           :', DEVICE)
print('  Mixed precision  :', use_amp)
print('  Max epochs       :', max_epochs)
print('  Early patience   :', patience)
print('  Monitor metric   :', monitor_metric, f"({monitor_mode})")
print('  Pos weights      :', {k: round(float(v), 3) for k, v in zip(target_cols, pos_weight_np)})
print('-' * 80)

for epoch in range(1, max_epochs + 1):
    t0 = time.time()

    train_loss = train_one_epoch(model, train_loader, optimizer, criterion, DEVICE, scaler, use_amp=use_amp)
    val_loss = evaluate_loss(model, val_loader, criterion, DEVICE, use_amp=use_amp)

    scheduler.step()
    lr = optimizer.param_groups[0]['lr']

    epoch_sec = time.time() - t0
    row = {
        'epoch': epoch,
        'train_loss': float(train_loss),
        'val_loss': float(val_loss),
        'lr': float(lr),
        'epoch_seconds': float(epoch_sec),
    }
    history.append(row)

    current_monitor = row['val_loss'] if monitor_metric == 'val_mean_brier' else row['val_loss']

    improved = is_better(current_monitor, best_val, monitor_mode)
    if improved:
        best_val = current_monitor
        best_epoch = epoch
        wait = 0
        best_state = deepcopy(model.state_dict())

        torch.save(
            {
                'epoch': epoch,
                'model_state_dict': best_state,
                'optimizer_state_dict': optimizer.state_dict(),
                'best_monitor_value': float(best_val),
                'monitor_metric': monitor_metric,
                'monitor_mode': monitor_mode,
                'model_cfg': model_cfg,
                'feature_cols': feature_cols,
                'target_cols': target_cols,
                'seed': SEED,
            },
            best_ckpt_path,
        )

    else:
        wait += 1

    print(
        f"Epoch {epoch:03d}/{max_epochs} | "
        f"train_loss={train_loss:.5f} | val_loss={val_loss:.5f} | "
        f"lr={lr:.6f} | {epoch_sec:.1f}s | "
        f"best_epoch={best_epoch:03d}"
    )

    if wait >= patience:
        print(f"Early stopping triggered at epoch {epoch}. No improvement for {patience} epochs.")
        break

# Restore best state before moving to evaluation
if best_state is not None:
    model.load_state_dict(best_state)

hist_df = pd.DataFrame(history)
hist_df.to_csv(train_log_path, index=False)

run_meta = {
    'timestamp': datetime.now(timezone.utc).isoformat(),
    'device': str(DEVICE),
    'use_amp': bool(use_amp),
    'seed': int(SEED),
    'profile_name': profile_name,
    'max_epochs': int(max_epochs),
    'early_stopping_patience': int(patience),
    'best_epoch': int(best_epoch),
    'best_monitor_value': float(best_val),
    'monitor_metric': monitor_metric,
    'monitor_mode': monitor_mode,
    'learning_rate': float(CFG['training']['learning_rate']),
    'weight_decay': float(CFG['training']['weight_decay']),
    'batch_size': int(batch_size),
    'num_workers': int(num_workers),
    'pos_weight': {k: float(v) for k, v in zip(target_cols, pos_weight_np)},
    'n_features': int(len(feature_cols)),
    'n_targets': int(len(target_cols)),
    'train_rows': int(len(df_train)),
    'val_rows': int(len(df_val)),
    'test_rows': int(len(df_test)),
}
with open(run_meta_path, 'w', encoding='utf-8') as f:
    json.dump(run_meta, f, indent=2)

print('\nTraining finished.')
print('Best epoch:', best_epoch)
print(f"Best checkpoint: {best_ckpt_path}")
print(f"Training history: {train_log_path}")
print(f"Run metadata: {run_meta_path}")

## 7) Evaluation Skeleton (AUC / F1 / Brier / ECE)

In [ ]:
# Step 6: Evaluation (AUC/F1/Brier/ECE + calibration curves)
import matplotlib.pyplot as plt
from sklearn.calibration import calibration_curve
from sklearn.metrics import roc_auc_score, f1_score

@torch.no_grad()
def collect_predictions(model, loader, device, use_amp=True):
    model.eval()
    probs_all, y_all = [], []

    for xb, yb in loader:
        xb = xb.to(device, non_blocking=True)
        with autocast_context(use_amp):
            logits = model(xb)
            probs = torch.sigmoid(logits)

        probs_all.append(probs.detach().cpu().numpy())
        y_all.append(yb.detach().cpu().numpy())

    return np.vstack(probs_all), np.vstack(y_all)


def expected_calibration_error(y_true, y_prob, n_bins=10):
    bins = np.linspace(0.0, 1.0, n_bins + 1)
    ece = 0.0
    for i in range(n_bins):
        left, right = bins[i], bins[i + 1]
        mask = (y_prob >= left) & (y_prob <= right) if i == n_bins - 1 else (y_prob >= left) & (y_prob < right)
        if mask.sum() == 0:
            continue
        acc = y_true[mask].mean()
        conf = y_prob[mask].mean()
        ece += (mask.mean()) * abs(acc - conf)
    return float(ece)


def evaluate_split(split_name, loader):
    probs, y_true = collect_predictions(model, loader, DEVICE, use_amp=use_amp)
    rows = []

    for idx, disease in enumerate(target_cols):
        yt = y_true[:, idx].astype(int)
        yp = probs[:, idx]
        yhat = (yp >= 0.5).astype(int)

        try:
            auc = roc_auc_score(yt, yp)
        except ValueError:
            auc = np.nan

        rows.append({
            'split': split_name,
            'disease': disease,
            'auc_roc': float(auc) if not np.isnan(auc) else np.nan,
            'f1': float(f1_score(yt, yhat, zero_division=0)),
            'brier': float(np.mean((yp - yt) ** 2)),
            'ece_10': float(expected_calibration_error(yt, yp, n_bins=int(CFG['evaluation']['calibration_bins']))),
            'positives': int(yt.sum()),
            'samples': int(len(yt)),
        })

    df = pd.DataFrame(rows)
    summary = {
        'split': split_name,
        'mean_auc_roc': float(df['auc_roc'].mean(skipna=True)),
        'mean_f1': float(df['f1'].mean()),
        'mean_brier': float(df['brier'].mean()),
        'mean_ece_10': float(df['ece_10'].mean()),
    }
    return probs, y_true, df, summary


val_probs, val_y, val_metrics_df, val_summary = evaluate_split('val', val_loader)
test_probs, test_y, test_metrics_df, test_summary = evaluate_split('test', test_loader)
all_metrics_df = pd.concat([val_metrics_df, test_metrics_df], ignore_index=True)
summary_df = pd.DataFrame([val_summary, test_summary])

val_metrics_path = metrics_dir / 'val_metrics_per_disease.csv'
test_metrics_path = metrics_dir / 'test_metrics_per_disease.csv'
summary_path = metrics_dir / 'summary_metrics.csv'

val_metrics_df.to_csv(val_metrics_path, index=False)
test_metrics_df.to_csv(test_metrics_path, index=False)
summary_df.to_csv(summary_path, index=False)

print('Validation summary:', val_summary)
print('Test summary:', test_summary)
print('\nPer-disease test metrics:')
show_df(test_metrics_df)

fig_dir = PROJECT_ROOT / CFG['artifacts']['figure_dir']
fig_dir.mkdir(parents=True, exist_ok=True)

n_targets = len(target_cols)
fig, axes = plt.subplots(1, n_targets, figsize=(5 * n_targets, 4), sharey=True)
if n_targets == 1:
    axes = [axes]

for i, disease in enumerate(target_cols):
    yt = test_y[:, i].astype(int)
    yp = test_probs[:, i]
    frac_pos, mean_pred = calibration_curve(yt, yp, n_bins=int(CFG['evaluation']['calibration_bins']), strategy='uniform')

    ax = axes[i]
    ax.plot(mean_pred, frac_pos, marker='o', label='FT-Transformer')
    ax.plot([0, 1], [0, 1], '--', color='gray', label='Perfect')
    ax.set_title(disease)
    ax.set_xlabel('Mean predicted probability')
    if i == 0:
        ax.set_ylabel('Fraction of positives')
    ax.grid(alpha=0.3)
    ax.legend(loc='best')

plt.tight_layout()
calib_path = fig_dir / 'test_calibration_curves_ft_transformer.png'
plt.savefig(calib_path, dpi=180)
plt.show()

print(f"Saved: {val_metrics_path}")
print(f"Saved: {test_metrics_path}")
print(f"Saved: {summary_path}")
print(f"Saved: {calib_path}")

## 8) Partial-Input Degradation Skeleton

In [ ]:
# Step 7: Partial-input degradation (kidney task) + optional Week6 XGBoost comparison
import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score

kidney_target = CFG['data']['target_columns']['kidney']
if kidney_target not in target_cols:
    raise ValueError(f"Kidney target '{kidney_target}' not found in target_cols: {target_cols}")

kidney_idx = target_cols.index(kidney_target)
phases = CFG['stress_test']['phases']

def affected_cols_by_keyword(columns, keyword):
    # Match base feature and its derived columns (e.g., *_mask)
    return [c for c in columns if c == keyword or c.startswith(f"{keyword}_")]


def apply_phase(X_df, phase, columns):
    X_mod = X_df.copy()

    if 'keep_only' in phase:
        keep_keywords = phase['keep_only']
        keep_cols = set()
        for key in keep_keywords:
            keep_cols.update(affected_cols_by_keyword(columns, key))

        zero_cols = [c for c in columns if c not in keep_cols]
        if zero_cols:
            X_mod.loc[:, zero_cols] = 0.0

    if 'remove' in phase:
        remove_keywords = phase['remove']
        remove_cols = set()
        for key in remove_keywords:
            remove_cols.update(affected_cols_by_keyword(columns, key))

        if remove_cols:
            X_mod.loc[:, list(remove_cols)] = 0.0

    return X_mod


def kidney_auc_from_df(model, X_df, y_df, batch_size=4096):
    ds = MultiDiseaseDataset(pd.concat([X_df, y_df], axis=1), feature_cols, target_cols)
    loader = DataLoader(ds, batch_size=batch_size, shuffle=False, num_workers=0, pin_memory=torch.cuda.is_available())
    probs, y_true = collect_predictions(model, loader, DEVICE, use_amp=use_amp)

    yk = y_true[:, kidney_idx].astype(int)
    pk = probs[:, kidney_idx]

    try:
        auc = roc_auc_score(yk, pk)
    except ValueError:
        auc = np.nan

    brier = float(np.mean((pk - yk) ** 2))
    ece = float(expected_calibration_error(yk, pk, n_bins=int(CFG['evaluation']['calibration_bins'])))
    return auc, brier, ece


X_test_df = df_test[feature_cols].copy()
y_test_df = df_test[target_cols].copy()

ft_rows = []
for phase in phases:
    phase_name = phase['name']
    X_phase = apply_phase(X_test_df, phase, feature_cols)
    auc, brier, ece = kidney_auc_from_df(model, X_phase, y_test_df)

    ft_rows.append({
        'phase': phase_name,
        'ft_auc_kidney': float(auc) if not np.isnan(auc) else np.nan,
        'ft_brier_kidney': float(brier),
        'ft_ece_kidney': float(ece),
    })

ft_deg_df = pd.DataFrame(ft_rows)
ft_deg_csv = metrics_dir / 'degradation_ft_kidney.csv'
ft_deg_df.to_csv(ft_deg_csv, index=False)
print('FT degradation metrics:')
show_df(ft_deg_df)
print(f"Saved: {ft_deg_csv}")

# Optional Week6 XGBoost reference loading (if a CSV exists in workspace)
xgb_ref_df = None
candidate_csvs = []
for pattern in [
    '**/*week6*degradation*.csv',
    '**/*xgboost*degradation*.csv',
    '**/*degradation*comparison*.csv',
]:
    candidate_csvs.extend(PROJECT_ROOT.glob(pattern))

for p in candidate_csvs:
    try:
        tmp = pd.read_csv(p)
    except Exception:
        continue

    cols = {c.lower(): c for c in tmp.columns}
    has_phase = any('phase' in c for c in cols)
    has_auc = any('auc' in c and 'kidney' in c for c in cols) or ('auc' in cols)
    if has_phase and has_auc:
        xgb_ref_df = tmp.copy()
        print(f"Using Week6 reference file: {p}")
        break

compare_df = ft_deg_df.copy()
if xgb_ref_df is not None:
    phase_col = [c for c in xgb_ref_df.columns if 'phase' in c.lower()][0]
    auc_candidates = [c for c in xgb_ref_df.columns if ('auc' in c.lower() and 'kidney' in c.lower())]
    if not auc_candidates:
        auc_candidates = [c for c in xgb_ref_df.columns if c.lower() == 'auc']

    if auc_candidates:
        xgb_auc_col = auc_candidates[0]
        xgb_small = xgb_ref_df[[phase_col, xgb_auc_col]].rename(columns={phase_col: 'phase', xgb_auc_col: 'xgb_auc_kidney'})
        compare_df = compare_df.merge(xgb_small, on='phase', how='left')

compare_csv = metrics_dir / 'degradation_comparison_ft_vs_xgb.csv'
compare_df.to_csv(compare_csv, index=False)
print(f"Saved: {compare_csv}")

# Plot comparison figure (required artifact name)
fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(compare_df['phase'], compare_df['ft_auc_kidney'], marker='o', linewidth=2, label='FT-Transformer (Week7)')

if 'xgb_auc_kidney' in compare_df.columns:
    ax.plot(compare_df['phase'], compare_df['xgb_auc_kidney'], marker='s', linewidth=2, label='XGBoost Sentinel (Week6)')
else:
    print('Week6 XGBoost degradation CSV not found; plotted FT curve only.')

ax.set_title('Kidney Partial-Input Degradation: FT vs XGBoost')
ax.set_xlabel('Degradation phase')
ax.set_ylabel('Kidney AUC-ROC')
ax.set_ylim(0.0, 1.0)
ax.grid(alpha=0.3)
ax.legend(loc='best')
plt.xticks(rotation=20, ha='right')
plt.tight_layout()

deg_fig_path = fig_dir / 'degradation_comparison_ft_vs_xgb.png'
plt.savefig(deg_fig_path, dpi=180)
plt.show()
print(f"Saved: {deg_fig_path}")

## 9) Artifact Export Skeleton

In [ ]:
# Step 8: Artifact export + requirement checks + Colab-friendly bundle
import zipfile

# Build required canonical files from outputs of Steps 5-7
metrics_per_disease_path = metrics_dir / 'metrics_per_disease.csv'
calibration_summary_path = metrics_dir / 'calibration_summary.csv'

if 'all_metrics_df' in globals():
    all_metrics_df.to_csv(metrics_per_disease_path, index=False)
    print(f"Saved: {metrics_per_disease_path}")
else:
    print('Warning: all_metrics_df not found. Run Step 6 first.')

if 'summary_df' in globals():
    summary_df.to_csv(calibration_summary_path, index=False)
    print(f"Saved: {calibration_summary_path}")
else:
    print('Warning: summary_df not found. Run Step 6 first.')

required_outputs = CFG['artifacts']['required_outputs']

# Map required names to concrete paths
artifact_map = {
    'best_model.pt': model_dir / 'best_model.pt',
    'metrics_per_disease.csv': metrics_per_disease_path,
    'calibration_summary.csv': calibration_summary_path,
    'degradation_comparison_ft_vs_xgb.png': fig_dir / 'degradation_comparison_ft_vs_xgb.png',
    'run_metadata.json': metrics_dir / 'run_metadata.json',
}

status_rows = []
for name in required_outputs:
    path = artifact_map.get(name, None)
    exists = bool(path is not None and path.exists())
    status_rows.append({'artifact': name, 'path': str(path) if path is not None else 'N/A', 'exists': exists})

status_df = pd.DataFrame(status_rows)
print('Required artifact status:')
display(status_df)

missing = status_df.loc[~status_df['exists'], 'artifact'].tolist()
if missing:
    print('Missing required artifacts:', missing)
else:
    print('All required artifacts are present ✅')

# Create a single zip bundle for easy Colab download
bundle_path = metrics_dir / 'week7_ft_transformer_artifacts.zip'
with zipfile.ZipFile(bundle_path, 'w', compression=zipfile.ZIP_DEFLATED) as zf:
    for row in status_rows:
        p = Path(row['path']) if row['path'] != 'N/A' else None
        if p is not None and p.exists():
            arcname = p.relative_to(PROJECT_ROOT)
            zf.write(p, arcname=str(arcname))

    # Include useful extra files if they exist
    extras = [
        metrics_dir / 'train_history.csv',
        metrics_dir / 'summary_metrics.csv',
        metrics_dir / 'val_metrics_per_disease.csv',
        metrics_dir / 'test_metrics_per_disease.csv',
        metrics_dir / 'degradation_ft_kidney.csv',
        metrics_dir / 'degradation_comparison_ft_vs_xgb.csv',
        fig_dir / 'test_calibration_curves_ft_transformer.png',
        CONFIG_PATH,
    ]
    for p in extras:
        if p.exists():
            arcname = p.relative_to(PROJECT_ROOT)
            zf.write(p, arcname=str(arcname))

print(f"Bundle created: {bundle_path}")
print('If running in Colab, download with:')
print("from google.colab import files; files.download('" + str(bundle_path).replace('\\\\','/') + "')")